<a href="https://colab.research.google.com/github/Viditp25/flyrank-intern-/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This notebook evaluates whether historical content-performance signals and available content metadata can rank content items labelled as experiencing an impressions decline. It compares three classifiers against the Week 4 rule-based baseline on one held-out client-grouped test set.

**Important interpretation:** `trend_direction == "down"` is the observed target label. The task is predictive decision support, not proof of causation. This is a client-grouped holdout, not a time-based validation. Predictor availability before the outcome period must be verified from the data-generation process.

## 1. Method choice and data validation

The target is binary: `1` means the source label is `down`; `0` means any other accepted label. Decision Tree offers simple nonlinear rules, Logistic Regression provides interpretable directional coefficients, and Random Forest can capture nonlinear interactions.

Precision@K measures the share of declining items among the top K recommendations. Average Precision and ROC-AUC assess ranking/discrimination across thresholds. Brier Score evaluates probability quality for classifiers; it is not applicable to the deterministic Week 4 rule because that rule does not produce calibrated probabilities.

In [8]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
)

RANDOM_STATE = 42
DATA_REL = Path("data/raw/content_refresh_anonymized.csv")
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "Viditp25/flyrank-intern-/main/"
    "data/raw/content_refresh_anonymized.csv"
)


def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / DATA_REL).is_file():
            return candidate
    return None


repo_root = find_repo_root()
data_source = str(repo_root / DATA_REL) if repo_root else DATA_URL
df = pd.read_csv(data_source)

required_columns = [
    "client_id", "content_id", "trend_direction", "days_since_last_update",
    "impressions_90d", "impressions_last_30d", "impressions_prev_30d",
    "clicks_90d", "clicks_last_30d", "clicks_prev_30d",
    "sessions_90d", "sessions_last_30d", "sessions_prev_30d",
]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(
        f"Dataset is missing required columns: {missing_columns}. "
        "The Week 4 baseline requires days_since_last_update; verify the exact source field if absent."
    )

print(f"Dataset source: {data_source}")
print(f"Rows: {len(df):,} | Columns: {len(df.columns):,}")
print("Column names:", df.columns.tolist())
display(df.head())

labels = df["trend_direction"].astype("string").str.strip().str.lower()
print("Observed trend_direction labels:")
display(labels.value_counts(dropna=False).to_frame("rows"))

valid_labels = {"down", "stable", "up", "new", "flat"}
invalid_labels = labels.isna() | ~labels.isin(valid_labels)
if invalid_labels.any():
    raise ValueError(
        "Missing or unexpected target labels found:\n"
        f"{labels[invalid_labels].value_counts(dropna=False)}"
    )

df["target"] = labels.eq("down").astype(int)
if df["target"].nunique() != 2:
    raise ValueError("The binary target must contain both classes.")

print("Binary target distribution:")
display(df["target"].value_counts().sort_index().rename_axis("target").to_frame("rows"))
print(f"Observed decline-label rate: {df['target'].mean():.2%}")

Dataset source: https://raw.githubusercontent.com/Viditp25/flyrank-intern-/main/data/raw/content_refresh_anonymized.csv
Rows: 30,000 | Columns: 44
Column names: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


Observed trend_direction labels:


,rows
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152


Binary target distribution:


,rows
target,
0,13738
1,16262


Observed decline-label rate: 54.21%


### Historical-window reconstruction

For impressions, clicks, and sessions, the days 61–90 value is reconstructed as the 90-day total minus the latest 30-day and preceding 30-day values. This assumes the three windows are aligned, non-overlapping, and sum to the reported 90-day total. Missing source values and negative reconstructions are recorded and set to missing rather than treated as valid counts.

In [9]:
audit_rows = []
for metric in ["impressions", "clicks", "sessions"]:
    total = pd.to_numeric(df[f"{metric}_90d"], errors="coerce")
    last = pd.to_numeric(df[f"{metric}_last_30d"], errors="coerce")
    previous = pd.to_numeric(df[f"{metric}_prev_30d"], errors="coerce")
    older = total - last - previous
    missing_source = total.isna() | last.isna() | previous.isna()
    negative_reconstruction = older.lt(0) & ~missing_source
    invalid_reconstruction = missing_source | negative_reconstruction

    df[f"{metric}_days_61_90"] = older.mask(invalid_reconstruction)
    audit_rows.append({
        "Metric": metric,
        "Rows": len(df),
        "Missing source rows": int(missing_source.sum()),
        "Negative reconstructed rows": int(negative_reconstruction.sum()),
        "Valid reconstructed rows": int((~invalid_reconstruction).sum()),
    })

audit_table = pd.DataFrame(audit_rows)
print("Historical reconstruction audit:")
display(audit_table)

# Alias used by the original Week 4 baseline.
df["impr_days_61_90"] = df["impressions_days_61_90"]

Historical reconstruction audit:


,Metric,Rows,Missing source rows,Negative reconstructed rows,Valid reconstructed rows
0,impressions,30000,0,0,30000
1,clicks,30000,0,0,30000
2,sessions,30000,0,0,30000


## 2. Feature engineering and client-grouped split

Features are constructed from the preceding 30-day and 61–90-day windows. The latest 30-day values, full 90-day totals, target labels, trend labels, and identifiers are excluded from model predictors. Missing-value imputation, scaling, and one-hot encoding are fitted within each model pipeline using training data only.

A fixed-seed 80/20 client-grouped split keeps all content belonging to a client in one partition. This reduces client-level leakage, but does not replace a time-based validation.

In [10]:
# Keep rows with both grouping ID and content ID.
model_df = df.loc[df["client_id"].notna() & df["content_id"].notna()].copy()

# Build preceding-window totals and derived historical features.
for metric in ["impressions", "clicks", "sessions"]:
    previous = pd.to_numeric(model_df[f"{metric}_prev_30d"], errors="coerce")
    older = pd.to_numeric(model_df[f"{metric}_days_61_90"], errors="coerce")
    model_df[f"{metric}_pre"] = previous + older

impressions_pre = pd.to_numeric(model_df["impressions_pre"], errors="coerce")
clicks_pre = pd.to_numeric(model_df["clicks_pre"], errors="coerce")
model_df["ctr_pre"] = clicks_pre / impressions_pre.replace(0, np.nan)

older_impressions = pd.to_numeric(model_df["impressions_days_61_90"], errors="coerce")
previous_impressions = pd.to_numeric(model_df["impressions_prev_30d"], errors="coerce")
model_df["prior_momentum_pct"] = (
    (previous_impressions - older_impressions)
    / older_impressions.replace(0, np.nan)
) * 100

log_source_columns = [
    "impressions_days_61_90", "clicks_days_61_90", "sessions_days_61_90",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "impressions_pre", "clicks_pre", "sessions_pre",
]
for col in log_source_columns:
    values = pd.to_numeric(model_df[col], errors="coerce")
    model_df[f"log1p_{col}"] = np.log1p(values.where(values >= 0))

numeric_candidates = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_days_61_90", "clicks_days_61_90", "sessions_days_61_90",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "impressions_pre", "clicks_pre", "sessions_pre", "ctr_pre", "prior_momentum_pct",
    "log1p_impressions_days_61_90", "log1p_clicks_days_61_90", "log1p_sessions_days_61_90",
    "log1p_impressions_prev_30d", "log1p_clicks_prev_30d", "log1p_sessions_prev_30d",
    "log1p_impressions_pre", "log1p_clicks_pre", "log1p_sessions_pre",
]
categorical_candidates = ["competition_level", "content_type", "main_intent", "word_count_tier"]

forbidden_features = {
    "client_id", "content_id", "trend_direction", "target", "trend_pct",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_90d", "clicks_90d", "sessions_90d",
}
numeric_features = [
    col for col in numeric_candidates
    if col in model_df.columns and col not in forbidden_features
]
categorical_features = [
    col for col in categorical_candidates
    if col in model_df.columns and col not in forbidden_features
]

for col in numeric_features:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
numeric_features = [col for col in numeric_features if model_df[col].notna().any()]
feature_cols = numeric_features + categorical_features
if not feature_cols:
    raise ValueError("No eligible predictor columns were found. Check the dataset schema.")

X = model_df[feature_cols].copy()
y = model_df["target"].astype(int).copy()
groups = model_df["client_id"].copy()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
transformers = []
if numeric_features:
    transformers.append(("num", numeric_pipeline, numeric_features))
if categorical_features:
    transformers.append(("cat", categorical_pipeline, categorical_features))
pre = ColumnTransformer(transformers=transformers, remainder="drop")

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

metadata_cols = [
    col for col in ["content_id", "client_id", "main_intent", "word_count_tier"]
    if col in model_df.columns
]
train_meta = model_df.iloc[train_idx][metadata_cols].copy()
test_meta = model_df.iloc[test_idx][metadata_cols].copy()
train_clients = set(train_meta["client_id"].dropna())
test_clients = set(test_meta["client_id"].dropna())
client_overlap = train_clients.intersection(test_clients)

assert not client_overlap, f"Client overlap found: {len(client_overlap)}"
assert y_train.nunique() == 2 and y_test.nunique() == 2, "Both classes must occur in train and test."
assert len(y_test) >= 100, "At least 100 test rows are required for Precision@100."

print(f"Rows used: {len(model_df):,}")
print(f"Training rows: {len(X_train):,} | Test rows: {len(X_test):,}")
print(f"Training clients: {len(train_clients):,} | Test clients: {len(test_clients):,}")
print(f"Client overlap: {len(client_overlap)}")
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Training target distribution:")
display(y_train.value_counts().sort_index().rename_axis("target").to_frame("rows"))
print("Test target distribution:")
display(y_test.value_counts().sort_index().rename_axis("target").to_frame("rows"))

Rows used: 30,000
Training rows: 23,837 | Test rows: 6,163
Training clients: 25 | Test clients: 7
Client overlap: 0
Numeric features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_days_61_90', 'clicks_days_61_90', 'sessions_days_61_90', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'impressions_pre', 'clicks_pre', 'sessions_pre', 'ctr_pre', 'prior_momentum_pct', 'log1p_impressions_days_61_90', 'log1p_clicks_days_61_90', 'log1p_sessions_days_61_90', 'log1p_impressions_prev_30d', 'log1p_clicks_prev_30d', 'log1p_sessions_prev_30d', 'log1p_impressions_pre', 'log1p_clicks_pre', 'log1p_sessions_pre']
Categorical features: ['competition_level', 'content_type', 'main_intent', 'word_count_tier']
Training target distribution:


,rows
target,
0,10724
1,13113


Test target distribution:


,rows
target,
0,3014
1,3149


## 3. Train and compare models against the Week 4 baseline

The three classifiers and the original Week 4 rule are evaluated using the same held-out test rows and labels. The Week 4 rule is: eligible pages must have `days_since_last_update >= 91` and `impr_days_61_90 >= 100`; eligible pages are ranked by `impr_days_61_90`. The thresholds are assignment-rule thresholds, not validated production thresholds.

**Baseline field check:** the original rule requires `days_since_last_update`. If the dataset does not contain that exact field, this notebook stops with a clear error. Verify the field name and meaning against the Week 4 notebook; do not substitute a different variable without evidence.

For the rule baseline, Precision@K is calculated only among eligible pages. If fewer than K pages are eligible, that Precision@K is reported as unavailable because the rule cannot provide K recommendations. The baseline's ROC-AUC and Average Precision use the complete test set, with ineligible pages assigned a score below eligible pages. The baseline does not produce calibrated probabilities, so its Brier Score is not applicable.

In [11]:
# Validate exact inputs required by the Week 4 rule.
baseline_required = ["days_since_last_update", "impr_days_61_90"]
missing_baseline_columns = [col for col in baseline_required if col not in model_df.columns]
if missing_baseline_columns:
    raise KeyError(
        "Cannot reproduce the Week 4 baseline because these exact columns are missing: "
        f"{missing_baseline_columns}. Check the Week 4 notebook and dataset schema; "
        "do not replace them with guessed aliases."
    )

# Precision@K for a ranked list; tied scores are resolved deterministically by content ID.
def precision_at_k(y_true, scores, content_ids, ks=(10, 20, 50, 100)):
    ranked = pd.DataFrame({
        "actual": np.asarray(y_true, dtype=int),
        "score": np.asarray(scores, dtype=float),
        "content_id": np.asarray(content_ids).astype(str),
    }).sort_values(["score", "content_id"], ascending=[False, True], kind="mergesort")
    if len(ranked) < max(ks):
        raise ValueError(f"At least {max(ks)} test rows are required; got {len(ranked)}.")
    return {f"Precision@{k}": float(ranked["actual"].iloc[:k].mean()) for k in ks}


def evaluate_classifier(y_true, probabilities, content_ids):
    y_array = np.asarray(y_true, dtype=int)
    probability_array = np.asarray(probabilities, dtype=float)
    if len(y_array) != len(probability_array):
        raise ValueError("Labels and predicted probabilities have different lengths.")
    if not np.isfinite(probability_array).all() or not ((probability_array >= 0) & (probability_array <= 1)).all():
        raise ValueError("Predicted probabilities must be finite and between 0 and 1.")
    result = precision_at_k(y_array, probability_array, content_ids)
    result["ROC-AUC"] = roc_auc_score(y_array, probability_array)
    result["Average Precision"] = average_precision_score(y_array, probability_array)
    result["Brier Score"] = brier_score_loss(y_array, probability_array)
    return result


models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=3, min_samples_leaf=50, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Logistic Regression": LogisticRegression(
        C=0.1, max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=8, min_samples_leaf=20,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ),
}

fitted = {}
predictions = {}
model_rows = []
for name, estimator in models.items():
    pipeline = Pipeline([
        ("preprocessor", clone(pre)),
        ("model", clone(estimator)),
    ])
    pipeline.fit(X_train, y_train)
    probabilities = pipeline.predict_proba(X_test)[:, 1]
    fitted[name] = pipeline
    predictions[name] = probabilities
    model_rows.append({
        "Method": name,
        **evaluate_classifier(y_test, probabilities, test_meta["content_id"]),
    })

# Apply the original Week 4 eligibility rule to the test rows only.
baseline_test = model_df.iloc[test_idx].copy()
baseline_age = pd.to_numeric(baseline_test["days_since_last_update"], errors="coerce")
baseline_impressions = pd.to_numeric(baseline_test["impr_days_61_90"], errors="coerce")
baseline_eligible = baseline_age.ge(91) & baseline_impressions.ge(100)
eligible_count = int(baseline_eligible.sum())
ks = (10, 20, 50, 100)

# Scores below every eligible score keep eligible pages ahead for full-test ranking metrics.
baseline_scores_all = np.where(
    baseline_eligible.to_numpy(),
    baseline_impressions.fillna(0).to_numpy(dtype=float),
    -1.0,
)

baseline_metrics = {}
eligible_ranked = pd.DataFrame({
    "actual": np.asarray(y_test, dtype=int)[baseline_eligible.to_numpy()],
    "score": baseline_impressions.loc[baseline_eligible].to_numpy(dtype=float),
    "content_id": test_meta.loc[baseline_eligible.to_numpy(), "content_id"].astype(str).to_numpy(),
}).sort_values(["score", "content_id"], ascending=[False, True], kind="mergesort")

for k in ks:
    if eligible_count >= k:
        baseline_metrics[f"Precision@{k}"] = float(eligible_ranked["actual"].iloc[:k].mean())
    else:
        baseline_metrics[f"Precision@{k}"] = np.nan

baseline_metrics["ROC-AUC"] = roc_auc_score(y_test, baseline_scores_all)
baseline_metrics["Average Precision"] = average_precision_score(y_test, baseline_scores_all)
baseline_metrics["Brier Score"] = np.nan
model_rows.append({"Method": "Week 4 Rule Baseline", **baseline_metrics})

comparison = pd.DataFrame(model_rows).set_index("Method")
comparison = comparison.sort_values("Average Precision", ascending=False, na_position="last")

print(f"Week 4 baseline eligible test pages: {eligible_count:,} / {len(y_test):,} ({eligible_count / len(y_test):.2%})")
print("Model vs Week 4 Rule Baseline:")
display(comparison.round(4))

# Validate comparison structure and values. Baseline Precision@K can be NA only when its eligible queue is too short.
expected_models = {"Decision Tree", "Logistic Regression", "Random Forest"}
expected_methods = expected_models | {"Week 4 Rule Baseline"}
assert set(comparison.index) == expected_methods
assert len(comparison) == 4
ranking_columns = ["Precision@10", "Precision@20", "Precision@50", "Precision@100", "ROC-AUC", "Average Precision"]
for method in expected_models:
    assert np.isfinite(comparison.loc[method, ranking_columns].to_numpy(dtype=float)).all()
    assert np.isfinite(comparison.loc[method, "Brier Score"])
assert np.isfinite(comparison.loc["Week 4 Rule Baseline", ["ROC-AUC", "Average Precision"]].to_numpy(dtype=float)).all()
assert pd.isna(comparison.loc["Week 4 Rule Baseline", "Brier Score"])
for k in ks:
    value = comparison.loc["Week 4 Rule Baseline", f"Precision@{k}"]
    assert (eligible_count >= k and np.isfinite(value)) or (eligible_count < k and pd.isna(value))

print("PASS: Three classifiers and the Week 4 rule were evaluated on the same held-out test rows.")

Week 4 baseline eligible test pages: 730 / 6,163 (11.84%)
Model vs Week 4 Rule Baseline:


,Precision@10,Precision@20,Precision@50,Precision@100,ROC-AUC,Average Precision,Brier Score
Method,,,,,,,
Random Forest,0.6,0.50,0.48,0.54,0.6581,0.6096,0.2204
Logistic Regression,0.8,0.75,0.66,0.66,0.6171,0.5988,0.2505
Decision Tree,1.0,0.65,0.66,0.65,0.6426,0.5975,0.2235
Week 4 Rule Baseline,0.3,0.25,0.38,0.34,0.4919,0.5028,NaN


PASS: Three classifiers and the Week 4 rule were evaluated on the same held-out test rows.


## 4. Error analysis and interpretation

Logistic Regression is used for a detailed diagnostic because its coefficients provide directional associations in the encoded feature space. Coefficients are not causal effects. The 0.5 probability threshold is used only for error analysis; it is not assumed to be the best operational threshold for ranking.

In [12]:
# 1. Inspect Logistic Regression coefficients.
lr = fitted["Logistic Regression"]
lr_probabilities = predictions["Logistic Regression"]
feature_names = lr.named_steps["preprocessor"].get_feature_names_out()
coefficients = lr.named_steps["model"].coef_[0]
assert len(feature_names) == len(coefficients)

coefficient_table = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Absolute coefficient": np.abs(coefficients),
}).sort_values("Absolute coefficient", ascending=False)
print("Top 15 Logistic Regression coefficients (by absolute magnitude):")
display(coefficient_table.head(15).round(4))

# 2. Classification errors at a diagnostic threshold of 0.5.
threshold = 0.5
errors = test_meta[["content_id"]].copy().reset_index(drop=True)
errors["actual"] = np.asarray(y_test, dtype=int)
errors["probability"] = np.asarray(lr_probabilities, dtype=float)
errors["predicted"] = (errors["probability"] >= threshold).astype(int)
errors["error_type"] = np.select(
    [
        (errors["actual"] == 1) & (errors["predicted"] == 0),
        (errors["actual"] == 0) & (errors["predicted"] == 1),
    ],
    ["False Negative", "False Positive"],
    default="Correct",
)
print("Classification outcome counts:")
display(errors["error_type"].value_counts().rename_axis("error_type").to_frame("count"))

false_positives = errors[errors["error_type"] == "False Positive"].sort_values("probability", ascending=False)
false_negatives = errors[errors["error_type"] == "False Negative"].sort_values("probability", ascending=True)
print("Highest-probability false positives:")
display(false_positives[["content_id", "actual", "probability"]].head(5))
print("Lowest-probability false negatives:")
display(false_negatives[["content_id", "actual", "probability"]].head(5))

# 3. Subgroup audit; small groups are omitted to avoid over-interpreting unstable estimates.
for col in ["main_intent", "word_count_tier"]:
    if col not in test_meta.columns:
        print(f"Subgroup audit skipped: {col} is unavailable.")
        continue
    subgroup = test_meta[[col]].copy().reset_index(drop=True)
    subgroup["actual"] = np.asarray(y_test, dtype=int)
    subgroup["predicted"] = errors["predicted"].to_numpy()
    subgroup["correct"] = (subgroup["actual"] == subgroup["predicted"]).astype(int)
    summary = (
        subgroup.groupby(col, dropna=False)
        .agg(N=("actual", "size"), decline_rate=("actual", "mean"), accuracy=("correct", "mean"))
        .reset_index()
    )
    summary = summary.loc[summary["N"] >= 30]
    print(f"Subgroup results for {col} (N >= 30):")
    display(summary.round(4))

# 4. Confusion matrix.
tn, fp, fn, tp = confusion_matrix(errors["actual"], errors["predicted"], labels=[0, 1]).ravel()
print("Confusion matrix counts at threshold 0.5:")
print(f"True negatives:  {tn}")
print(f"False positives: {fp}")
print(f"False negatives: {fn}")
print(f"True positives:  {tp}")
assert len(errors) == len(y_test)
assert (tn + fp + fn + tp) == len(y_test)
print("PASS: Error analysis completed and aligns with the held-out test labels.")

Top 15 Logistic Regression coefficients (by absolute magnitude):


,Feature,Coefficient,Absolute coefficient
22,num__log1p_impressions_pre,1.4970,1.4970
16,num__log1p_impressions_days_61_90,-0.8641,0.8641
41,cat__word_count_tier_Unknown,-0.8138,0.8138
23,num__log1p_clicks_pre,-0.7638,0.7638
19,num__log1p_impressions_prev_30d,0.7396,0.7396
21,num__log1p_sessions_prev_30d,-0.6474,0.6474
24,num__log1p_sessions_pre,0.4934,0.4934
37,cat__word_count_tier_1000-2000,0.4857,0.4857
32,cat__main_intent_Unknown,0.4374,0.4374
8,num__impressions_prev_30d,-0.3943,0.3943


Classification outcome counts:


,count
error_type,
Correct,3500
False Positive,1512
False Negative,1151


Highest-probability false positives:


...

Subgroup results for main_intent (N >= 30):


,main_intent,N,decline_rate,accuracy
0,commercial,986,0.5000,0.5680
1,informational,4000,0.4962,0.5605
3,transactional,1152,0.5729,0.5920


Subgroup results for word_count_tier (N >= 30):


,word_count_tier,N,decline_rate,accuracy
0,1000-2000,806,0.4404,0.5682
1,2000-3500,3863,0.5423,0.5713
2,3500+,409,0.4841,0.5721
3,NaN,1085,0.4618,0.5539


Confusion matrix counts at threshold 0.5:
True negatives:  1502
False positives: 1512
False negatives: 1151
True positives:  1998
PASS: Error analysis completed and aligns with the held-out test labels.


## 5. Conclusion

The conclusion below is generated from the metrics computed in this notebook run. Review the output after executing all cells; do not copy results from an earlier run. Precision@K is most relevant when review capacity is limited. Average Precision and ROC-AUC summarize broader ranking/discrimination performance. Brier Score is available only for the three probabilistic classifiers. The Week 4 baseline may have unavailable Precision@K values if its eligibility rule returns fewer than K pages.

In [13]:
# Build a conclusion from the current run's comparison table (no hard-coded old metrics).
model_comparison = comparison.loc[["Decision Tree", "Logistic Regression", "Random Forest"]]
best_ap_model = model_comparison["Average Precision"].idxmax()
best_auc_model = model_comparison["ROC-AUC"].idxmax()
best_brier_model = model_comparison["Brier Score"].idxmin()

print("CONCLUSION — CURRENT NOTEBOOK RUN")
print(f"Highest Average Precision among classifiers: {best_ap_model} ({model_comparison.loc[best_ap_model, 'Average Precision']:.4f}).")
print(f"Highest ROC-AUC among classifiers: {best_auc_model} ({model_comparison.loc[best_auc_model, 'ROC-AUC']:.4f}).")
print(f"Lowest Brier Score among classifiers: {best_brier_model} ({model_comparison.loc[best_brier_model, 'Brier Score']:.4f}).")
for k in (10, 20, 50, 100):
    metric = f"Precision@{k}"
    best_model = model_comparison[metric].idxmax()
    best_value = model_comparison.loc[best_model, metric]
    print(f"Highest {metric} among classifiers: {best_model} ({best_value:.4f}).")

baseline = comparison.loc["Week 4 Rule Baseline"]
print(f"Week 4 baseline eligibility: {eligible_count} of {len(y_test)} test pages.")
for metric in ["Precision@10", "Precision@20", "Precision@50", "Precision@100", "ROC-AUC", "Average Precision"]:
    value = baseline[metric]
    if pd.isna(value):
        print(f"Week 4 baseline {metric}: unavailable (fewer than K eligible pages).")
    else:
        print(f"Week 4 baseline {metric}: {value:.4f}.")

print("Interpretation: choose a method based on the review capacity and the metric that matches the intended workflow. These results describe predictive performance on a client-grouped holdout, not causation. A later time-based evaluation and verification of predictor availability are needed before operational use.")

CONCLUSION — CURRENT NOTEBOOK RUN
Highest Average Precision among classifiers: Random Forest (0.6096).
Highest ROC-AUC among classifiers: Random Forest (0.6581).
Lowest Brier Score among classifiers: Random Forest (0.2204).
Highest Precision@10 among classifiers: Decision Tree (1.0000).
Highest Precision@20 among classifiers: Logistic Regression (0.7500).
Highest Precision@50 among classifiers: Decision Tree (0.6600).
Highest Precision@100 among classifiers: Logistic Regression (0.6600).
Week 4 baseline eligibility: 730 of 6163 test pages.
Week 4 baseline Precision@10: 0.3000.
Week 4 baseline Precision@20: 0.2500.
Week 4 baseline Precision@50: 0.3800.
Week 4 baseline Precision@100: 0.3400.
Week 4 baseline ROC-AUC: 0.4919.
Week 4 baseline Average Precision: 0.5028.
Interpretation: choose a method based on the review capacity and the metric that matches the intended workflow. These results describe predictive performance on a client-grouped holdout, not causation. A later time-based eval

## Self-check before submission

The automated checks below validate the executed notebook state. They cannot verify that the notebook has been committed to GitHub or prove that every feature was historically available. Complete those manual checks yourself after running **Runtime → Run all** from a clean kernel.

- [ ] Run all cells from a clean kernel without errors.
- [ ] Confirm the Week 4 baseline fields and their meanings match the original Week 4 notebook.
- [ ] Confirm historical-window reconstruction assumptions against the dataset documentation.
- [ ] Confirm predictor availability before the prediction period, or retain the limitation in the report.
- [ ] Review the current-run comparison table and conclusion.
- [ ] Save and commit as `work/notebooks/w05_model.ipynb`.

In [14]:
# FINAL AUTOMATED VALIDATION

required_models = {"Decision Tree", "Logistic Regression", "Random Forest"}
expected_methods = required_models | {"Week 4 Rule Baseline"}
required_metrics = [
    "Precision@10", "Precision@20", "Precision@50", "Precision@100",
    "ROC-AUC", "Average Precision", "Brier Score",
]

assert "comparison" in globals(), "Run Section 3 before the final self-check."
assert set(comparison.index) == expected_methods, "Comparison must include all three classifiers and the Week 4 baseline."
assert all(metric in comparison.columns for metric in required_metrics), "A required metric is missing."
assert len(comparison) == 4

# Classifier metrics must all be finite; baseline P@K can be NA only when fewer than K pages qualify.
for model_name in required_models:
    assert np.isfinite(comparison.loc[model_name, required_metrics].to_numpy(dtype=float)).all(), f"Invalid metrics for {model_name}."
for k in (10, 20, 50, 100):
    value = comparison.loc["Week 4 Rule Baseline", f"Precision@{k}"]
    assert (eligible_count >= k and np.isfinite(value)) or (eligible_count < k and pd.isna(value))
assert np.isfinite(comparison.loc["Week 4 Rule Baseline", ["ROC-AUC", "Average Precision"]].to_numpy(dtype=float)).all()
assert pd.isna(comparison.loc["Week 4 Rule Baseline", "Brier Score"])

assert set(predictions) == required_models
for model_name, scores in predictions.items():
    scores = np.asarray(scores, dtype=float)
    assert len(scores) == len(y_test), f"Prediction count mismatch for {model_name}."
    assert np.isfinite(scores).all() and ((scores >= 0) & (scores <= 1)).all(), f"Invalid probabilities for {model_name}."

assert train_clients.isdisjoint(test_clients), "Client overlap detected."
assert len(errors) == len(y_test), "Error analysis row count mismatch."
assert set(errors["error_type"].unique()).issubset({"Correct", "False Positive", "False Negative"})
assert (tn + fp + fn + tp) == len(y_test)

print("=" * 58)
print("FINAL NOTEBOOK SELF-CHECK: ALL AUTOMATED CHECKS PASSED")
print("PASS: Three classifiers and Week 4 baseline are compared.")
print("PASS: Required ranking metrics and classifier Brier Scores are valid.")
print("PASS: Baseline unavailable metrics are handled explicitly.")
print("PASS: Predictions align with the held-out test rows.")
print("PASS: No client overlap between train and test.")
print("PASS: Error analysis aligns with the test labels.")
print("MANUAL: Verify historical feature availability and commit the notebook.")
print("=" * 58)

FINAL NOTEBOOK SELF-CHECK: ALL AUTOMATED CHECKS PASSED
PASS: Three classifiers and Week 4 baseline are compared.
PASS: Required ranking metrics and classifier Brier Scores are valid.
PASS: Baseline unavailable metrics are handled explicitly.
PASS: Predictions align with the held-out test rows.
PASS: No client overlap between train and test.
PASS: Error analysis aligns with the test labels.
MANUAL: Verify historical feature availability and commit the notebook.
